In [1]:
from pyspark.sql.functions import col, when, to_date, coalesce, lit, udf, trim
from pyspark.sql.types import IntegerType

# 1. Load Bronze Table
bronze = spark.table("raw_ltfs_loans")

# 2. UDF to parse strings like '2yrs 3mon' or '5yrs 11mon' into total months
def parse_duration_to_months(duration_str):
    if duration_str is None:
        return 0
    try:
        parts = str(duration_str).strip().split()
        years = int(parts[0].replace("yrs", "").replace("yr", "")) if len(parts) > 0 else 0
        months = 0
        if len(parts) > 1:
            months = int(parts[1].replace("mon", "").replace("mths", "").replace("mth", ""))
        return (years * 12) + months
    except Exception:
        return 0

parse_duration_udf = udf(parse_duration_to_months, IntegerType())



StatementMeta(, 4e292bcf-2afd-4954-bc8c-b9d8345eac68, 3, Finished, Available, Finished, False)

In [2]:
# 3. Clean and transform columns
silver = (
    bronze
    .withColumnRenamed("UniqueID", "loan_id")
    .withColumnRenamed("Date.of.Birth", "dob_raw")
    .withColumnRenamed("Employment.Type", "employment_type")
    .withColumnRenamed("DisbursalDate", "disbursal_date_raw")
    .withColumnRenamed("PERFORM_CNS.SCORE", "cibil_score")
    .withColumnRenamed("PERFORM_CNS.SCORE.DESCRIPTION", "cibil_description")
    .withColumnRenamed("PRI.NO.OF.ACCTS", "pri_total_accts")
    .withColumnRenamed("PRI.ACTIVE.ACCTS", "pri_active_accts")
    .withColumnRenamed("PRI.OVERDUE.ACCTS", "pri_overdue_accts")
    .withColumnRenamed("PRI.CURRENT.BALANCE", "pri_current_balance")
    .withColumnRenamed("PRI.SANCTIONED.AMOUNT", "pri_sanctioned_amt")
    .withColumnRenamed("PRI.DISBURSED.AMOUNT", "pri_disbursed_amt")
    .withColumnRenamed("SEC.NO.OF.ACCTS", "sec_total_accts")
    .withColumnRenamed("SEC.ACTIVE.ACCTS", "sec_active_accts")
    .withColumnRenamed("SEC.OVERDUE.ACCTS", "sec_overdue_accts")
    .withColumnRenamed("SEC.CURRENT.BALANCE", "sec_current_balance")
    .withColumnRenamed("SEC.SANCTIONED.AMOUNT", "sec_sanctioned_amt")
    .withColumnRenamed("SEC.DISBURSED.AMOUNT", "sec_disbursed_amt")
    .withColumnRenamed("PRIMARY.INSTAL.AMT", "pri_instal_amt")
    .withColumnRenamed("SEC.INSTAL.AMT", "sec_instal_amt")
    .withColumnRenamed("NEW.ACCTS.IN.LAST.SIX.MONTHS", "new_accts_6m")
    .withColumnRenamed("DELINQUENT.ACCTS.IN.LAST.SIX.MONTHS", "delinquent_accts_6m")
    .withColumnRenamed("AVERAGE.ACCT.AGE", "avg_acct_age_raw")
    .withColumnRenamed("CREDIT.HISTORY.LENGTH", "credit_history_raw")
    .withColumnRenamed("NO.OF_INQUIRIES", "no_of_inquiries")
    .withColumnRenamed("Current_pincode_ID", "pincode_id")
    .withColumnRenamed("State_ID", "state_id")
    .withColumnRenamed("Aadhar_flag", "aadhaar_flag")
    .withColumnRenamed("loan_default", "is_default")
)



StatementMeta(, 4e292bcf-2afd-4954-bc8c-b9d8345eac68, 4, Finished, Available, Finished, False)

In [3]:
silver_cleaned = (
    silver
    .withColumn("disbursal_date", coalesce(
        to_date(col("disbursal_date_raw"), "dd-MM-yy"),
        to_date(col("disbursal_date_raw"), "dd-MM-yyyy")
    ))
    .withColumn("date_of_birth", coalesce(
        to_date(col("dob_raw"), "dd-MM-yy"),
        to_date(col("dob_raw"), "dd-MM-yyyy")
    ))
    .withColumn("employment_type", coalesce(trim(col("employment_type")), lit("Unemployed/Unknown")))
    .withColumn("avg_acct_age_months", parse_duration_udf(col("avg_acct_age_raw")))
    .withColumn("credit_history_months", parse_duration_udf(col("credit_history_raw")))
    .withColumn("total_active_accts", col("pri_active_accts") + col("sec_active_accts"))
    .withColumn("total_overdue_accts", col("pri_overdue_accts") + col("sec_overdue_accts"))
    .withColumn("total_current_balance", col("pri_current_balance") + col("sec_current_balance"))
    .withColumn("kyc_count",
        col("aadhaar_flag") + col("PAN_flag") + col("VoterID_flag") + col("Driving_flag") + col("Passport_flag")
    )
    .withColumn("cibil_band",
        when(col("cibil_score") <= 0, "No History (NTC)")
        .when(col("cibil_score") < 550, "Very Poor (300-549)")
        .when(col("cibil_score") < 650, "Poor (550-649)")
        .when(col("cibil_score") < 750, "Fair/Good (650-749)")
        .otherwise("Excellent (750+)")
    )
    .withColumn("ltv_band",
        when(col("ltv") >= 85.0, "High LTV (>=85%)")
        .when(col("ltv") >= 70.0, "Medium LTV (70-85%)")
        .otherwise("Low LTV (<70%)")
    )
    .withColumn("overdue_ratio",
        when(col("total_active_accts") > 0, col("total_overdue_accts") / col("total_active_accts"))
        .otherwise(lit(0.0))
    )
    .dropDuplicates(["loan_id"])
)



StatementMeta(, 4e292bcf-2afd-4954-bc8c-b9d8345eac68, 5, Finished, Available, Finished, False)

In [4]:
# 4. Save Silver Table
silver_cleaned.write.mode("overwrite").format("delta").saveAsTable("silver_ltfs_loans")

print(f"✅ Silver table created successfully: {silver_cleaned.count():,} rows")

StatementMeta(, 4e292bcf-2afd-4954-bc8c-b9d8345eac68, 6, Finished, Available, Finished, False)

✅ Silver table created successfully: 233,154 rows
